In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier

df = pd.read_csv("email.csv")
df["Message"] = df["Message"].fillna("").astype(str)
df["y"] = (df["Category"].str.lower() == "spam").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    df["Message"], df["y"], test_size=0.2, random_state=42, stratify=df["y"]
)

# Embedding pipeline: TF-IDF -> SVD (dense embedding) -> classifier
embed_model = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, stop_words="english", max_features=10000)),
    ("svd", TruncatedSVD(n_components=200, random_state=42)),  # embeddings
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(64, 32),   # ← hidden layers
        activation="relu",             # ← activation
        solver="adam",                 # ← optimizer
        learning_rate_init=0.001,
        max_iter=50,
        random_state=42
    ))
])

embed_model.fit(X_train, y_train)

emails = [
    "WIN a free prize now!!! click link",
    "Share the address to send goodies"
]
probs = embed_model.predict_proba(emails)[:, 1]
for e, p in zip(emails, probs):
    print("\nEMAIL:", e)
    print("Spam probability:", float(p))



EMAIL: WIN a free prize now!!! click link
Spam probability: 0.9954160829652933

EMAIL: Share the address to send goodies
Spam probability: 0.004149152950844058


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(
